In [27]:
                                    #  Part 4

In [28]:
# Task 4.1

In [29]:
!pip install -q google-genai pydantic

In [30]:
from google import genai
from google.genai import types
import json
from getpass import getpass

API_KEY = getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini client initialized successfully!")

Paste your Gemini API key: ··········
Gemini client initialized successfully!


In [45]:
# Defining the JSON schema

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {
            "type": "string"
        },
        "confidence": {
            "type": "number"
        },
        "summary": {
            "type": "string"
        },
        "recommended_action": {
            "type": "string"
        }
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

print("JSON schema created successfully!")

JSON schema created successfully!


In [46]:
# Defining the system instruction and CRAFT prompt

SYSTEM_TEXT = """
You are a careful healthcare operations analyst.
Analyse patient follow-up notes accurately.
Do not invent information or assume reasons that are not stated.
Return only the requested structured JSON response.
"""

ALLOWED_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

def create_prompt(note_text):
    prompt = f"""
Context:
CareLoop Clinics wants to understand why patients miss appointments.

Role:
You are a healthcare operations analyst.

Action:
Read the following patient follow-up note and identify the main reason
for missing the appointment.

Allowed root_cause values:
{ALLOWED_CAUSES}

If the reason is unclear or does not fit the categories, use "other".
Do not guess.

Tone:
Be factual, concise and neutral.

Format:
Return a JSON object with exactly these four fields:
1. root_cause: one allowed value
2. confidence: a number between 0 and 1
3. summary: maximum 25 words
4. recommended_action: a practical action for the clinic

Patient follow-up note:
{note_text}
"""
    return prompt

print("System instruction and CRAFT prompt created successfully!")

System instruction and CRAFT prompt created successfully!


In [47]:
import pandas as pd

notes_df = pd.read_csv("followup_notes.csv")

print("Dataset shape:", notes_df.shape)
display(notes_df.head())

# Retrieving note N001
note_001 = notes_df.loc[
    notes_df["note_id"] == "N001", "note_text"
].iloc[0]

print("\nNote N001:")
print(note_001)

Dataset shape: (30, 3)


,note_id,appointment_id,note_text
0,N001,A0872,Patient said she never got any reminder messag...
1,N002,A0897,Lives far from the clinic and could not find a...
2,N003,A0526,Patient felt the consultation fee was too high...
3,N004,A0926,NaN
4,N005,A0534,"Had high fever and body ache on the day, staye..."



Note N001:
Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.


In [48]:
MODEL_NAME = "gemini-3.5-flash"

def analyse_note(note_text):
    prompt = create_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    data = json.loads(response.text)
    return data

In [49]:
result_n001 = analyse_note(note_001)

print("N001 Analysis:")
print(result_n001)

N001 Analysis:
{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'The patient missed the appointment because she did not receive a reminder message and forgot about the scheduled visit.', 'recommended_action': "Verify the patient's contact details and confirm that automated appointment reminders are properly configured and sending."}


In [50]:
# Task 4.2

In [51]:
# Creating the Pydantic validation model

from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

print("Pydantic validation model created successfully!")

Pydantic validation model created successfully!


In [52]:
LAST_RAW_RESPONSE = ""

def analyse_note(note_text):
    global LAST_RAW_RESPONSE

    prompt = create_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    LAST_RAW_RESPONSE = response.text
    data = json.loads(response.text)
    return data

print("Function updated to preserve raw responses!")

Function updated to preserve raw responses!


In [53]:
# Processing the first 10 notes

import time

analysed_records = []

for i, row in notes_df.head(10).iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    print(f"Processing {note_id}...")

    # Handling blank notes
    if pd.isna(note_text) or not str(note_text).strip():

        record = {
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank",
            "raw_response": ""
        }

    else:
        data = {}
        raw_response = ""

        try:
            data = analyse_note(str(note_text))
            raw_response = LAST_RAW_RESPONSE

            # Validating response fields
            insight = FollowupInsight(**data)

            # Validating allowed root cause
            if insight.root_cause not in ALLOWED_CAUSES:
                raise ValueError("Invalid root_cause category")

            validation_status = "valid"

        except Exception as e:
            validation_status = "invalid"
            print(f"Validation issue for {note_id}: {e}")

        record = {
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": data.get("root_cause", ""),
            "confidence": data.get("confidence", ""),
            "summary": data.get("summary", ""),
            "recommended_action": data.get("recommended_action", ""),
            "validation_status": validation_status,
            "raw_response": raw_response
        }

        # Waiting between API calls
        if i < 9:
            time.sleep(13)

    analysed_records.append(record)

# Creating and saving the output DataFrame
analysed_df = pd.DataFrame(analysed_records)

analysed_df.to_csv("notes_analysed.csv", index=False)

print("\nProcessing completed!")
print("Dataset shape:", analysed_df.shape)

display(analysed_df)

Processing N001...
Processing N002...
Validation issue for N002: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 19h56m20.506900012s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Generate content requests per da

,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,0.98,The patient missed the visit because she did n...,"Verify the patient's contact details, send aut...",valid,"{""root_cause"":""forgot_or_no_reminder"",""confide..."
1,N002,A0897,,,,,invalid,
2,N003,A0526,,,,,invalid,
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,,,,,invalid,
5,N006,A0507,,,,,invalid,
6,N007,A0333,,,,,invalid,
7,N008,A0222,distance_or_transport,0.95,The patient missed the appointment due to a ci...,Offer telehealth options or arrange dedicated ...,valid,"{""root_cause"":""distance_or_transport"",""confide..."
8,N009,A0977,,,,,invalid,
9,N010,A0596,,,,,invalid,


In [54]:
check_df = pd.read_csv("notes_analysed.csv")

print("Shape:", check_df.shape)
print("\nValidation status counts:")
print(check_df["validation_status"].value_counts())

print("\nMissing values:")
print(check_df.isnull().sum())

print("\nN004 verification:")
display(check_df[check_df["note_id"] == "N004"])

Shape: (10, 8)

Validation status counts:
validation_status
invalid          7
valid            2
skipped_blank    1
Name: count, dtype: int64

Missing values:
note_id               0
appointment_id        0
root_cause            7
confidence            7
summary               7
recommended_action    7
validation_status     0
raw_response          8
dtype: int64

N004 verification:


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
3,N004,A0926,other,0.0,No note recorded,Re-call patient,skipped_blank,NaN


In [55]:
# Task 4.3

In [56]:
# Test 1: Invalid confidence value
bad_reply_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment.",
    "recommended_action": "Send reminders."
}

# Test 2: Missing recommended_action
bad_reply_2 = {
    "root_cause": "cost_or_fee",
    "confidence": 0.8,
    "summary": "Patient could not afford the fee."
}

for i, bad_reply in enumerate([bad_reply_1, bad_reply_2], start=1):
    try:
        FollowupInsight(**bad_reply)
        print(f"Test {i}: Validation passed")
    except ValidationError as e:
        print(f"\nTest {i}: Validation failed as expected")
        print(e)


Test 1: Validation failed as expected
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Test 2: Validation failed as expected
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'cost_or_f...ld not afford the fee.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [61]:
# Comparing V1 and V2 prompts

def compare_prompts(note_text, version):

    if version == "V1":
        prompt = f"Classify this note:\n{note_text}"

    else:
        prompt = create_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    data = json.loads(response.text)

    return {
        "root_cause": data.get("root_cause", ""),
        "is_allowed": data.get("root_cause", "") in ALLOWED_CAUSES
    }


comparison_results = []

for note_id in ["N007", "N009"]:

    note_text = notes_df.loc[
        notes_df["note_id"] == note_id, "note_text"
    ].iloc[0]

    for version in ["V1", "V2"]:

        print(f"Testing {version} on {note_id}...")

        result = compare_prompts(note_text, version)

        comparison_results.append({
            "note_id": note_id,
            "prompt_version": version,
            "root_cause": result["root_cause"],
            "is_allowed": result["is_allowed"]
        })

        time.sleep(13)

comparison_df = pd.DataFrame(comparison_results)

display(comparison_df)

In [62]:
calls_task_4_1 = 1
calls_task_4_2 = 9
calls_task_4_3 = 4

total_api_calls = (
    calls_task_4_1 +
    calls_task_4_2 +
    calls_task_4_3
)

print("Total Gemini API calls:", total_api_calls)

Total Gemini API calls: 14


In [63]:
# Task 4.4

In [64]:
# Merging two datasets

appointments_df = pd.read_csv("appointments_clean.csv")

# Merge with analysed follow-up notes
merged_df = pd.merge(
    analysed_df,
    appointments_df[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left"
)

print("Merged dataset shape:", merged_df.shape)

display(merged_df[
    ["note_id", "appointment_id", "root_cause",
     "reminder_sent", "validation_status"]
])

Merged dataset shape: (10, 9)


,note_id,appointment_id,root_cause,reminder_sent,validation_status
0,N001,A0872,forgot_or_no_reminder,No,valid
1,N002,A0897,,No,invalid
2,N003,A0526,,Yes,invalid
3,N004,A0926,other,No,skipped_blank
4,N005,A0534,,Yes,invalid
5,N006,A0507,,Yes,invalid
6,N007,A0333,,No,invalid
7,N008,A0222,distance_or_transport,Yes,valid
8,N009,A0977,,Yes,invalid
9,N010,A0596,,No,invalid


In [65]:
# Creating the cross-tabulation

cause_reminder_table = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)

print("Root Cause vs Reminder Sent:")
display(cause_reminder_table)

Root Cause vs Reminder Sent:


reminder_sent,No,Yes
root_cause,,
,3,4
distance_or_transport,0,1
forgot_or_no_reminder,1,0
other,1,0
